In [ ]:
from pathlib import Path

import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = Path("..") if NOTEBOOK_DIR.name == "statistic" else Path(".")
LOG_ROOT = PROJECT_ROOT / "logs2"
AUTOENCODER_LOG_ROOT = PROJECT_ROOT / "autoencoder" / "autoencoder_logs"


def latest_run(pattern: str) -> Path:
    runs = sorted(AUTOENCODER_LOG_ROOT.glob(pattern))
    if not runs:
        raise FileNotFoundError(f"No autoencoder runs match {pattern!r} in {AUTOENCODER_LOG_ROOT}")
    return runs[-1]


def read_scalar_series(run_path: Path, tag: str, max_step: int | None = None):
    if not run_path.exists():
        raise FileNotFoundError(f"Missing TensorBoard log directory: {run_path}")

    event_files = list(run_path.glob("events.out.tfevents*"))
    if not event_files:
        raise FileNotFoundError(f"No TensorBoard event file found in: {run_path}")

    accumulator = EventAccumulator(str(run_path), size_guidance={"scalars": 0})
    accumulator.Reload()

    scalar_tags = accumulator.Tags().get("scalars", [])
    if tag not in scalar_tags:
        raise KeyError(f"Tag {tag!r} not found in {run_path}. Available scalar tags: {scalar_tags}")

    series = [(point.step, point.value) for point in accumulator.Scalars(tag)]
    if max_step is not None:
        series = [(step, value) for step, value in series if step <= max_step]

    if not series:
        raise ValueError(f"No scalar points left after filtering {tag!r} to step <= {max_step}")

    return series


def load_curves(runs, tag: str, total_episodes: int | None = None, lower_is_better: bool = False):
    curves = []
    missing_runs = []

    for run in runs:
        try:
            series = read_scalar_series(run["path"], tag=tag, max_step=total_episodes)
        except Exception as exc:
            missing_runs.append((run["name"], run["path"], exc))
            continue

        key = (lambda item: item[1])
        best_step, best_value = min(series, key=key) if lower_is_better else max(series, key=key)
        curves.append({**run, "series": series, "best_step": best_step, "best_value": best_value})

    return curves, missing_runs


def print_curve_summary(curves, missing_runs, metric_name: str, lower_is_better: bool = False):
    best_word = "lowest" if lower_is_better else "best"
    for curve in curves:
        print(
            f"{curve['name']}: {len(curve['series'])} points, "
            f"{best_word} {metric_name} = {curve['best_value']:.4f} at episode {curve['best_step']}"
        )

    if missing_runs:
        print("\nRuns not plotted:")
        for name, path, exc in missing_runs:
            print(f"- {name}: {path} ({exc})")


def plot_curves(
    curves,
    y_label: str,
    info_lines,
    total_episodes: int | None = None,
    lower_is_better: bool = False,
    axis_width: float = 8.0,
    right_panel_width: float = 4.0,
    left_margin_width: float = 1.25,
    panel_gap_width: float = 0.25,
    height: float = 5.6,
):
    if not curves:
        raise RuntimeError("No curves were loaded. Check RUNS and log roots above.")

    fig_width = left_margin_width + axis_width + panel_gap_width + right_panel_width
    fig = plt.figure(figsize=(fig_width, height), dpi=140)

    bottom_margin = 0.72
    top_margin = 0.22
    plot_height = height - bottom_margin - top_margin

    ax = fig.add_axes([
        left_margin_width / fig_width,
        bottom_margin / height,
        axis_width / fig_width,
        plot_height / height,
    ])
    legend_ax = fig.add_axes([
        (left_margin_width + axis_width + panel_gap_width) / fig_width,
        bottom_margin / height,
        right_panel_width / fig_width,
        plot_height / height,
    ])
    legend_ax.axis("off")

    for idx, curve in enumerate(curves):
        steps = [step for step, _ in curve["series"]]
        values = [value for _, value in curve["series"]]
        ax.plot(steps, values, label=curve["name"], color=curve["color"], linewidth=2.2)
        ax.scatter(
            [curve["best_step"]],
            [curve["best_value"]],
            color=curve["color"],
            edgecolor="white",
            linewidth=1.2,
            s=70,
            zorder=5,
        )
        y_min, y_max = ax.get_ylim()
        baseline = y_max if lower_is_better else y_min
        ax.vlines(
            curve["best_step"],
            ymin=min(baseline, curve["best_value"]),
            ymax=max(baseline, curve["best_value"]),
            color=curve["color"],
            linestyle="--",
            linewidth=1.1,
            alpha=0.75,
        )
        ax.hlines(
            curve["best_value"],
            xmin=0,
            xmax=curve["best_step"],
            color=curve["color"],
            linestyle="--",
            linewidth=1.1,
            alpha=0.75,
        )
        ax.annotate(
            f"{curve['best_value']:.2f}" if not lower_is_better else f"{curve['best_value']:.4f}",
            xy=(0, curve["best_value"]),
            xytext=(-10, 5 if idx % 2 == 0 else -5),
            textcoords="offset points",
            ha="right",
            va="bottom" if idx % 2 == 0 else "top",
            color=curve["color"],
            fontsize=9,
            fontweight="bold",
            annotation_clip=False,
        )
        ax.annotate(
            f"{curve['best_step']}",
            xy=(curve["best_step"], ax.get_ylim()[0]),
            xytext=(0, -28),
            textcoords="offset points",
            ha="center",
            va="top",
            color=curve["color"],
            fontsize=8,
            fontweight="bold",
            annotation_clip=False,
        )

    legend_x = 0.03
    legend_y = 0.98
    line_width = 0.09
    text_gap = 0.035
    line_gap = 0.070
    row_gap = 0.018
    legend_width = 0.88

    row_heights = [line_gap * max(1, len(curve["name"].splitlines())) for curve in curves]
    legend_height = sum(row_heights) + row_gap * (len(curves) - 1) + 0.055
    legend_box = FancyBboxPatch(
        (0.0, legend_y - legend_height + 0.02),
        legend_width,
        legend_height,
        boxstyle="round,pad=0.012",
        facecolor="white",
        edgecolor="0.8",
        transform=legend_ax.transAxes,
        clip_on=False,
        zorder=10,
    )
    legend_ax.add_patch(legend_box)

    current_y = legend_y - 0.035
    for curve in curves:
        label_lines = curve["name"].splitlines()
        legend_ax.plot(
            [legend_x, legend_x + line_width],
            [current_y, current_y],
            transform=legend_ax.transAxes,
            color=curve["color"],
            linewidth=2.8,
            clip_on=False,
            zorder=11,
        )
        legend_ax.text(
            legend_x + line_width + text_gap,
            current_y,
            label_lines[0],
            transform=legend_ax.transAxes,
            ha="left",
            va="center",
            fontsize=10,
            zorder=11,
        )
        for line_idx, label_line in enumerate(label_lines[1:], start=1):
            legend_ax.text(
                legend_x + line_width + text_gap,
                current_y - line_gap * line_idx,
                label_line,
                transform=legend_ax.transAxes,
                ha="left",
                va="center",
                fontsize=10,
                zorder=11,
            )
        current_y -= line_gap * max(1, len(label_lines)) + row_gap

    info_y = legend_y - legend_height - 0.045
    for line_idx, info_line in enumerate(info_lines):
        legend_ax.text(
            legend_x,
            info_y - 0.085 * line_idx,
            info_line,
            transform=legend_ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
        )

    ax.set_xlabel("Training Episode", labelpad=10)
    ax.set_ylabel(y_label, labelpad=12)
    if total_episodes is not None:
        ax.set_xlim(0, total_episodes)
        ax.set_xticks(range(0, total_episodes + 1, 1000))
    else:
        ax.set_xlim(left=0)
    if not lower_is_better:
        ax.set_ylim(bottom=0)
    ax.grid(axis="x", linestyle="--", alpha=0.28)
    ax.grid(axis="y", linestyle="-", alpha=0.18)

    return fig, ax


print(f"Project root: {PROJECT_ROOT}")
print(f"DQN log root: {LOG_ROOT}")
print(f"Autoencoder log root: {AUTOENCODER_LOG_ROOT}")

: 

In [ ]:
AUTOENCODER_LOSS_TAG = "Loss/train"

RUNS = [
    {
        "name": "Autoencoder env11 v1",
        "path": latest_run("autoencoder_env11_variant0_v1_*"),
        "color": "tab:blue",
    },
    {
        "name": "Autoencoder env11 v2",
        "path": latest_run("autoencoder_env11_variant0_v2_*"),
        "color": "tab:orange",
    },
    {
        "name": "Autoencoder env11 v3",
        "path": latest_run("autoencoder_env11_variant0_v3_*"),
        "color": "tab:green",
    },
]

curves, missing_runs = load_curves(RUNS, tag=AUTOENCODER_LOSS_TAG, lower_is_better=True)
print_curve_summary(curves, missing_runs, metric_name="loss", lower_is_better=True)
fig, ax = plot_curves(
    curves,
    y_label="Training Loss",
    info_lines=[
        "Variant: 0",
        "Environment: 11",
        "Metric: Training Loss",
    ],
    lower_is_better=True,
    axis_width=8.0,
    right_panel_width=4.0,
)
plt.show()

In [ ]:
TOTAL_EPISODES = 10_000
VALIDATION_REWARD_TAG = "Reward/validation"

RUNS = [
    {
        "name": "DQN v11.11.1",
        "path": LOG_ROOT / "DQN_v11.11.1_variant_0",
        "color": "tab:blue",
    },
    {
        "name": "DQN v11.11.2",
        "path": LOG_ROOT / "DQN_v11.11.2_variant_0",
        "color": "tab:orange",
    },
    {
        "name": "DQN v11.11.4",
        "path": LOG_ROOT / "DQN_v11.11.4_variant_0",
        "color": "tab:green",
    },
    {
        "name": "DQN v11.11.6",
        "path": LOG_ROOT / "DQN_v11.11.6_variant_0",
        "color": "tab:red",
    },
]

curves, missing_runs = load_curves(RUNS, tag=VALIDATION_REWARD_TAG, total_episodes=TOTAL_EPISODES)
print_curve_summary(curves, missing_runs, metric_name="validation reward")
fig, ax = plot_curves(
    curves,
    y_label="Average Validation Reward",
    info_lines=[
        "Variant: 0",
        f"Total Episodes: {TOTAL_EPISODES // 1000}k",
        "DRL Policy: Rainbow DQN",
    ],
    total_episodes=TOTAL_EPISODES,
    axis_width=8.0,
    right_panel_width=4.0,
)
plt.show()

In [ ]:
TOTAL_EPISODES = 10_000
VALIDATION_REWARD_TAG = "Reward/validation"

RUNS = [
    {
        "name": "DQN v11.11.1",
        "path": LOG_ROOT / "DQN_v11.11.1_variant_0",
        "color": "tab:blue",
    },
    {
        "name": "DQN v11.11.3",
        "path": LOG_ROOT / "DQN_v11.11.3_variant_0",
        "color": "tab:orange",
    },
    {
        "name": "DQN v11.11.5",
        "path": LOG_ROOT / "DQN_v11.11.5_variant_0",
        "color": "tab:green",
    },
    {
        "name": "DQN v11.11.7",
        "path": LOG_ROOT / "DQN_v11.11.7_variant_0",
        "color": "tab:red",
    },
]

curves, missing_runs = load_curves(RUNS, tag=VALIDATION_REWARD_TAG, total_episodes=TOTAL_EPISODES)
print_curve_summary(curves, missing_runs, metric_name="validation reward")
fig, ax = plot_curves(
    curves,
    y_label="Average Validation Reward",
    info_lines=[
        "Variant: 0",
        f"Total Episodes: {TOTAL_EPISODES // 1000}k",
        "DRL Policy: Rainbow DQN",
    ],
    total_episodes=TOTAL_EPISODES,
    axis_width=8.0,
    right_panel_width=4.0,
)
plt.show()